# Timestamp 포함 데이터 재추출과 전체 불량 Shot 점검

대회용 CN7/RG3 라벨·비라벨 데이터의 센서값은 변경하지 않고, 원본 행 번호로 Timestamp와 제품 메타데이터를 붙입니다. RH/LH는 원래 행을 보존합니다. 불량 표에서만 같은 시각의 좌우 행을 한 Shot으로 묶습니다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

root = Path.cwd()
if not (root / 'data').exists():
    root = root.parent

data_dir = root / 'data'
raw_labeled = pd.read_csv(data_dir / 'original/moldset_labeled.csv')
raw_labeled['product'] = raw_labeled['PART_NAME'].str.extract(r'^(CN7|RG3)', expand=False)
raw_labeled['side'] = raw_labeled['PART_NAME'].str.extract(r'(LH|RH)\s*$', expand=False)

unlabeled_files = {
    product: pd.read_csv(data_dir / ('moldset_unlabeled_' + product.lower() + '.csv'))
    for product in ('CN7', 'RG3')
}
wanted_ids = set().union(*(set(frame['Unnamed: 0']) for frame in unlabeled_files.values()))
parts = []
for chunk in pd.read_csv(
    data_dir / 'original/unlabeled_data.csv',
    usecols=['Unnamed: 0', 'TimeStamp', 'PART_NAME', 'PART_NO', 'EQUIP_CD'],
    chunksize=100_000,
):
    parts.append(chunk.loc[chunk['Unnamed: 0'].isin(wanted_ids)])
raw_unlabeled_meta = pd.concat(parts, ignore_index=True)
assert len(raw_unlabeled_meta) == len(wanted_ids)
assert raw_unlabeled_meta['Unnamed: 0'].is_unique

source_meta = {
    'labeled': raw_labeled.set_index('Unnamed: 0'),
    'unlabeled': raw_unlabeled_meta.set_index('Unnamed: 0'),
}
outputs = {}
for kind in ('labeled', 'unlabeled'):
    for product in ('CN7', 'RG3'):
        file_name = 'moldset_' + kind + '_' + product.lower() + '.csv'
        competition = pd.read_csv(data_dir / file_name)
        assert competition['Unnamed: 0'].is_unique
        assert competition['Unnamed: 0'].isin(source_meta[kind].index).all()
        matched = source_meta[kind].loc[competition['Unnamed: 0']].reset_index(drop=True)
        assert matched['PART_NAME'].str.startswith(product).all()
        assert matched['TimeStamp'].notna().all()
        if kind == 'labeled':
            assert np.array_equal(matched['PassOrFail'], competition['PassOrFail'])
        meta_columns = ['TimeStamp', 'PART_NAME', 'PART_NO', 'EQUIP_CD']
        if kind == 'labeled':
            meta_columns += ['Reason']
        meta = matched[meta_columns].copy()
        meta['side'] = matched['PART_NAME'].str.extract(r'(LH|RH)\s*$', expand=False)
        if kind == 'labeled':
            meta['Reason'] = meta['Reason'].fillna('정상')
        result = pd.concat([
            competition[['Unnamed: 0']].rename(columns={'Unnamed: 0': 'source_row_id'}).reset_index(drop=True),
            meta.reset_index(drop=True),
            competition.drop(columns='Unnamed: 0').reset_index(drop=True),
        ], axis=1)
        result['TimeStamp'] = pd.to_datetime(result['TimeStamp'])
        result = result.sort_values(['TimeStamp', 'source_row_id']).reset_index(drop=True)
        target = data_dir / ('moldset_' + kind + '_' + product.lower() + '_with_timestamp.csv')
        result.to_csv(target, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d %H:%M:%S')
        outputs[(kind, product)] = result
        print(kind, product, 'rows=', len(result), 'unique_timestamps=', result['TimeStamp'].nunique(), 'saved=', target.name)

assert len(outputs[('labeled', 'CN7')]) == 1211
assert len(outputs[('labeled', 'RG3')]) == 1182
assert len(outputs[('unlabeled', 'CN7')]) == 35239
assert len(outputs[('unlabeled', 'RG3')]) == 35941

In [ ]:
# 대회 라벨 39 Shot + 원본에만 있는 초기 세팅 불량 5 Shot.
original = raw_labeled.copy()
original['TimeStamp'] = pd.to_datetime(original['TimeStamp'])
original['Reason'] = original['Reason'].fillna('정상')
competition_ids = set(outputs[('labeled', 'CN7')]['source_row_id']) | set(outputs[('labeled', 'RG3')]['source_row_id'])
original['in_competition'] = original['Unnamed: 0'].isin(competition_ids)

bad_rows = original.loc[original['PassOrFail'].eq(1)].copy()
bad_rows['side_reason'] = bad_rows['side'] + ' ' + bad_rows['Reason']
def joined(values):
    return '|'.join(sorted(set(values.dropna().astype(str))))
defect_shots = bad_rows.groupby(['product', 'TimeStamp'], as_index=False).agg(
    defect_type=('Reason', joined),
    bad_sides=('side', joined),
    side_reason=('side_reason', joined),
    in_competition=('in_competition', 'max'),
)
assert len(defect_shots) == 44
assert int(defect_shots['in_competition'].sum()) == 39

candidates = pd.read_csv(root / 'reports/tables/run_shot_feature_candidates.csv')
candidates['TimeStamp'] = pd.to_datetime(candidates['TimeStamp'])
candidate_index = candidates.set_index(['product', 'TimeStamp'])
assert candidate_index.index.is_unique

sensor_names = {
    'Max_Injection_Speed': '최대 사출속도',
    'Filling_Time': '충전시간',
    'Injection_Time': '사출시간',
    'Plasticizing_Time': '계량시간',
    'Cycle_Time': '사이클시간',
    'Max_Injection_Pressure': '최대 사출압력',
    'Max_Switch_Over_Pressure': '전환압력',
    'Max_Back_Pressure': '최대 배압',
    'Barrel_Temperature_5': '배럴온도5',
    'Hopper_Temperature': '호퍼온도',
    'Mold_Temperature_3': '금형온도3',
    'Mold_Temperature_4': '금형온도4',
}
# 작은 계측 흔들림을 '센서 튐'으로 부르지 않기 위한 탐색용 최소 변화량.
min_delta = {
    'Max_Injection_Speed': 1.0, 'Filling_Time': 0.08, 'Injection_Time': 0.08,
    'Plasticizing_Time': 0.30, 'Cycle_Time': 0.15,
    'Max_Injection_Pressure': 0.50, 'Max_Switch_Over_Pressure': 0.50,
    'Max_Back_Pressure': 0.50, 'Barrel_Temperature_5': 0.50,
    'Hopper_Temperature': 0.80, 'Mold_Temperature_3': 0.50,
    'Mold_Temperature_4': 0.50,
}
normal_abs_dev = {
    (product, feature): candidates.loc[
        candidates['product'].eq(product) & candidates['PassOrFail'].eq(0),
        feature + '__dev10',
    ].abs().dropna()
    for product in ('CN7', 'RG3') for feature in sensor_names
}
same_day_normal = original.loc[
    original['product'].eq('CN7') &
    original['TimeStamp'].dt.date.eq(pd.Timestamp('2020-10-27').date()) &
    original['PassOrFail'].eq(0)
].drop_duplicates('TimeStamp')
initial_reference = same_day_normal[['Max_Injection_Speed', 'Filling_Time']].median()
assert len(same_day_normal) == 102

records = []
for shot in defect_shots.itertuples(index=False):
    key = (shot.product, shot.TimeStamp)
    row = {
        'product': shot.product,
        'TimeStamp': shot.TimeStamp,
        'defect_type': shot.defect_type,
        'bad_sides': shot.bad_sides,
        'side_reason': shot.side_reason,
        'scope': '대회 라벨' if shot.in_competition else '원본만',
    }
    if shot.in_competition:
        current = candidate_index.loc[key]
        if isinstance(current, pd.DataFrame):
            raise ValueError('Duplicate competition shot: ' + str(key))
        row['run_id_estimated'] = current['run_id']
        row['observed_shot_seq'] = current['observed_shot_seq']
        found = []
        for feature, label in sensor_names.items():
            row[feature] = current[feature]
            deviation = current[feature + '__dev10']
            row[feature + '_past10_delta'] = deviation
            reference = normal_abs_dev[(shot.product, feature)]
            if pd.isna(deviation) or len(reference) < 30:
                continue
            tail = (int(reference.ge(abs(deviation)).sum()) + 1) / (len(reference) + 1)
            row[feature + '_normal_tail_fraction'] = tail
            if abs(deviation) >= min_delta[feature] and tail <= 0.01:
                found.append((feature, label, deviation, tail))
        found.sort(key=lambda t: abs(t[2]) / min_delta[t[0]], reverse=True)
        row['large_rare_sensor_changes'] = len(found)
        row['sensor_observation'] = (
            '; '.join('{} {} (직전 10샷 대비 {:+.2f})'.format(label, round(current[feature], 2), delta)
                      for feature, label, delta, _ in found[:3])
            if found else '선정한 12개 센서에서 뚜렷한 개별 이탈 없음'
        )
        row['comparison'] = '같은 제품의 라벨 정상 Shot에서 직전 10샷 대비 변화량과 비교'
    else:
        current = original.loc[
            original['product'].eq(shot.product) &
            original['TimeStamp'].eq(shot.TimeStamp)
        ].iloc[0]
        for feature in sensor_names:
            row[feature] = current[feature]
        speed_delta = current['Max_Injection_Speed'] - initial_reference['Max_Injection_Speed']
        fill_delta = current['Filling_Time'] - initial_reference['Filling_Time']
        row['large_rare_sensor_changes'] = pd.NA
        row['sensor_observation'] = (
            '최대 사출속도 {:.1f} (같은 날 정상 중앙값 대비 {:+.1f}); 충전시간 {:.2f} ({:+.2f})'
            .format(current['Max_Injection_Speed'], speed_delta, current['Filling_Time'], fill_delta)
        )
        row['comparison'] = '10/27 같은 날 나중에 관측된 정상 102 Shot의 중앙값과 비교'
    records.append(row)

diagnostics = pd.DataFrame(records).sort_values(['product', 'TimeStamp'])
table_path = root / 'reports/tables/all_defect_shot_diagnostics_competition_legacy.csv'
diagnostics.to_csv(table_path, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d %H:%M:%S')
print('Defect shots:', len(diagnostics), diagnostics.groupby(['product', 'defect_type', 'scope']).size().to_dict())
print('With clear individual sensor change:', diagnostics.groupby(['product', 'scope'])['large_rare_sensor_changes'].apply(lambda s: int(s.fillna(0).gt(0).sum())).to_dict())
print('Saved:', table_path)

In [ ]:
summary = [
    '# 모든 불량 Shot별 센서 관찰',
    '',
    '라벨 원본에서 RH/LH를 같은 Timestamp의 한 Shot으로 합쳤다. 대회 라벨 39 Shot과 참고용 원본에만 있는 초기허용불량 5 Shot, 총 44 Shot이다.',
    '',
    '대회 라벨 Shot의 관찰은 같은 제품 라벨 정상 Shot에서 계산한 직전 10개 관측 Shot 대비 변화량의 분포와 비교했다. 아래의 센서 변화 후보는 선정한 12개 센서 중 정상 기준 절대 변화량 상위 약 1%에 들고, 별도로 정한 최소 변화량도 넘는 경우다. 탐색적 기준이며 유의성 검정이나 분류 성능이 아니다. 직전 관측값에는 비라벨 데이터가 포함되지만, 비라벨을 정상이라고 가정하지 않았다.',
    '',
    '초기허용불량은 대회 라벨 범위에 없으며, 10/27 같은 날 이후 정상 Shot 102개의 중앙값과 비교했다. 이 비교는 시간상 미래 값을 사용하므로 설명용이지 예측 피처가 아니다.',
    '',
    '| 제품 | 시각 | 불량 유형 | 좌우별 불량 | 범위 | 센서 관찰 |',
    '|---|---|---|---|---|---|',
]
for shot in diagnostics.itertuples(index=False):
    summary.append('| {} | {} | {} | {} | {} | {} |'.format(
        shot.product,
        shot.TimeStamp.strftime('%m/%d %H:%M:%S'),
        shot.defect_type.replace('|', '·'),
        shot.side_reason.replace('|', '·'),
        shot.scope,
        shot.sensor_observation.replace('|', '·'),
    ))
summary += [
    '',
    '## 읽는 법',
    '',
    '- 뚜렷한 개별 이탈 없음: 선정한 센서·기준에서 해당 Shot 하나의 뚜렷한 변화가 없다는 뜻이다. 불량이 없다는 뜻이 아니다.',
    '- CN7 가스는 연속된 같은 사건에 집중되어 있어 Shot 하나의 센서 변화보다 시간적 집중 또는 Run 전체 조건이 더 중요할 수 있다.',
    '- RG3 가스·미성형은 이 기준에서 대부분 뚜렷한 개별 변화가 없다. 작은 변화나 여러 변수의 조합을 별도로 검토해야 한다.',
    '- RH/LH는 제품의 좌우 라벨이다. 실제 금형 캐비티 번호나 배치는 데이터만으로 확정할 수 없다.',
    '- Run ID는 연속 관측 간 30분 초과 공백 또는 제품 전환으로 나눈 추정 구간이며 실제 작업 로그가 아니다.',
    '- 센서값은 원본 단위로 제시했다. 별도 추출한 대회용 CSV의 센서값은 기존 표준화값을 그대로 유지한다.',
]
report_path = root / 'reports/all_defect_shot_summary_competition_legacy.md'
report_path.write_text('\n'.join(summary) + '\n', encoding='utf-8')
print('Saved:', report_path)
print('Rows in markdown:', len(diagnostics))